In [ ]:
import pandas as pd

import config as c

In [ ]:
# build PR-BD corpus

# 1) posts in BD subreddits
# subreddit_type based on categorisation here: https://github.com/glorisonne/reddit_bd_mood_posting_mh/blob/main/data/subreddit_topics.csv
# (Fourth level = "bipolar")
#posts = pd.read_csv(c.data + "posts_meta.csv", usecols=["id", "user_id", "subreddit_name", "text_wordcount", "mentions_bd",
#                                                        "subreddit_type"])

#posts = posts[posts.subreddit_type == "bd"]
#print("Posts in BD subreddits:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 2) posts that mention BD -- refer to BD terms list https://github.com/glorisonne/reddit_bd_user_characteristics/blob/master/disclosure-patterns/condition-terms/bipolar-filter-terms.txt
#posts = posts[posts.mentions_bd]
#print("Posts that mention BD:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 3) posts with at least 94 words, duplicates (same text by same user) removed
#posts = posts[posts.text_wordcount > 93]

# add post texts - read post_text csv in two batches as the file is very large and the code migth therefore fail on
# machines with less RAM
#posts_text_0 = pd.read_csv(c.data + "posts_text.csv", nrows=10000000)
#posts_text_0 = posts_text_0[posts_text_0.id.isin(posts.id)]
#posts_text_1 = pd.read_csv(c.data + "posts_text.csv", skiprows=10000000, header=None, names=["id", "text"])
#posts_text_1 = posts_text_1[posts_text_1.id.isin(posts.id)]
#posts_text = pd.concat([posts_text_0, posts_text_1])

#posts = posts.merge(posts_text, left_on="id", right_on="id")

#del(posts_text_0, posts_text_1, posts_text)

#posts.drop_duplicates(subset=["text", "user_id"], keep = "last", inplace=True)
#print("Posts with at least 94 words, duplicates removed:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))
posts.to_csv(c.data_local + "posts_bd.csv")

# TBD lemmatize (check when did this for the wordcount in the flowchart)
# preprocess_posts_with_spacy.py. nohup python process_posts_with_spacy.py data/posts_bd.csv &> output/process_posts_spacy.log &
comBine output files

# 4) cosine similarity with PR terms list - share PR terms list in data/ (ToDo: possibly share "standard" list with variants + list of only unique terms)
PR_scoring.py --> outputs posts_bd_spacy_phrases.csv und posts_bd_PR_scored.py

# 5) Reference corpus - cosine similarity cutoff

In [ ]:
# figure out where different number of posts comes from for BD subreddit posts

# lower/uppercasing does not matter for the BD subreddits in subreddit_topics.csv
subreddits = pd.read_csv(c.data_local + "subreddit_topics.csv")
bd_subreddits = subreddits[subreddits["fourth level"] == "bipolar"]
posts_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits.subreddit.str.lower())]
posts_with_case = posts[posts.subreddit_name.isin(bd_subreddits.subreddit)]
print(len(posts_ignore_case), len(posts_with_case))

# but it does matter for the subreddits from bipolar-subreddits.txt
bd_subreddits_old = pd.read_csv(c.data_local + "bipolar-subreddits.txt", header=None, names=["subreddit"]).squeeze(axis=0)
print(len(bd_subreddits_old))
print(bd_subreddits)
posts_bd_old = posts[posts.subreddit_name.isin(bd_subreddits_old.subreddit)]
posts_bd_old_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits_old.subreddit.str.lower())]
print(len(posts_bd_old), len(posts_bd_old_ignore_case))

# reason: three subreddits have different upper/lowercasing in bipolar-subreddits.txt
print(bd_subreddits[~bd_subreddits.subreddit.isin(bd_subreddits_old.subreddit)].subreddit)
print(bd_subreddits_old[~bd_subreddits_old.subreddit.isin(bd_subreddits.subreddit)].subreddit)

In [ ]:
# check unique PR terms
PR_terms = pd.read_csv(c.data_local + "PR_terms_unique.csv")
#print(PR_terms)
PR_terms.variants.fillna("", inplace=True)
PR_terms["num_variants"] = PR_terms.variants.apply(lambda x: len(x.split(", ")))
print(PR_terms.num_variants.sum())

# separate row for each variant
PR_terms["variants"] = PR_terms.variants.apply(lambda x: x.split(", "))
PR_terms_expanded = PR_terms.explode('variants').reset_index(drop=True)
import numpy as np
PR_terms_expanded["variants"] = np.where(PR_terms_expanded.variants.str.len() == 0, PR_terms_expanded.term,
                                          PR_terms_expanded.variants)
#pd.set_option('display.max_rows', 500)
#print(PR_terms_expanded[["term", "variants"]].head(n=500))

# PR_terms_expanded_orig = pd.read_csv(c.data_local + "PR_terms.csv")
# print(PR_terms_expanded_orig)
# PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace(" - ", "-")
#PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace(" #", "#")
#PR_terms_expanded_orig["term"] = PR_terms_expanded_orig.term.str.replace("# ", "#")
# validate # yourself # is missing in orig
print(PR_terms_expanded[~PR_terms_expanded.variants.isin(PR_terms_expanded_orig.term)].variants.unique())
print(PR_terms_expanded_orig[~PR_terms_expanded_orig.term.isin(PR_terms_expanded.variants)].term.unique())

In [ ]:
# calculate keyness + select key lemmas